# Kvasir-VQA-x1 Baseline Inference (BLIP-2)

Zero-shot VQA inference using a pre-trained BLIP-2 model on the Kvasir-VQA-x1 dataset.

**Prerequisites:**
- GPU Runtime enabled (`Runtime > Change runtime type > T4 GPU`)
- Project data uploaded to Google Drive OR download fresh in Colab

## 1. Install Dependencies

In [ ]:
!pip install -q datasets transformers accelerate pillow pandas tqdm

## 2. Configuration

Set `USE_DRIVE = True` if you uploaded the project to Google Drive.
Set `USE_DRIVE = False` to download everything fresh.

In [ ]:
import os

# ========== CHANGE THIS ==========
USE_DRIVE = True
# =================================

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # UPDATE THIS PATH to match your Google Drive folder
    PROJECT_DIR = "/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector"
else:
    PROJECT_DIR = "/content/medical-vqa"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
IMAGE_DIR = os.path.join(DATA_DIR, "images")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results", "predictions")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")
print(f"Data dir:    {DATA_DIR}")
print(f"Image dir:   {IMAGE_DIR}")

## 3. Download Dataset (Only if USE_DRIVE = False)

Skip this cell if you already have the data on Google Drive.

In [ ]:
if not USE_DRIVE:
    import json
    import pandas as pd
    from datasets import load_dataset
    from tqdm.auto import tqdm

    os.makedirs(IMAGE_DIR, exist_ok=True)

    # Download images
    print("[INFO] Downloading images from SimulaMet-HOST/Kvasir-VQA...")
    ds_host = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
    seen = set()
    for row in tqdm(ds_host, desc="Saving images"):
        if row["img_id"] not in seen:
            row["image"].save(os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg"))
            seen.add(row["img_id"])
    print(f"[INFO] Saved {len(seen)} images")

    # Download QA pairs
    for split in ["train", "test"]:
        print(f"[INFO] Downloading {split} split...")
        ds = load_dataset("SimulaMet/Kvasir-VQA-x1", split=split)
        records = []
        for row in tqdm(ds, desc=f"Processing {split}"):
            records.append({
                "img_id": row["img_id"],
                "complexity": row["complexity"],
                "question": row["question"],
                "answer": row["answer"],
                "question_class": row["question_class"],
                "original": json.dumps(row["original"]) if row.get("original") else ""
            })
        df = pd.DataFrame(records)
        df.to_csv(os.path.join(DATA_DIR, f"kvasir_vqa_x1_{split}.csv"), index=False)
        print(f"[INFO] Saved {len(df)} {split} samples")
else:
    print("[INFO] Using data from Google Drive. Skipping download.")

## 4. Load Model

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

MODEL_NAME = "Salesforce/blip2-opt-2.7b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 64
NUM_SAMPLES = 20

print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print(f"\nLoading model: {MODEL_NAME}")
processor = Blip2Processor.from_pretrained(MODEL_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
model = model.to(DEVICE)
model.eval()
print("Model loaded successfully!")

## 5. Select Diverse Test Samples

In [ ]:
import pandas as pd

test_csv = os.path.join(DATA_DIR, "kvasir_vqa_x1_test.csv")
test_df = pd.read_csv(test_csv)
print(f"Test set: {len(test_df)} samples")

# Select diverse samples across complexity levels
samples = []
for complexity in sorted(test_df["complexity"].unique()):
    subset = test_df[test_df["complexity"] == complexity]
    n = max(1, NUM_SAMPLES // 3)
    samples.append(subset.sample(n=min(n, len(subset)), random_state=42))
sample_df = pd.concat(samples).head(NUM_SAMPLES)

print(f"Selected {len(sample_df)} samples:")
print(sample_df["complexity"].value_counts().sort_index().to_string())

## 6. Run Inference

In [ ]:
from PIL import Image
from tqdm.auto import tqdm

results = []
correct = 0
total = 0

print(f"Running inference on {len(sample_df)} samples...\n")
print("=" * 80)

for idx, (_, row) in enumerate(sample_df.iterrows()):
    img_path = os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg")

    if not os.path.exists(img_path):
        print(f"[SKIP] Image not found: {row['img_id']}")
        continue

    image = Image.open(img_path).convert("RGB")
    question = row["question"]
    ground_truth = str(row["answer"])

    # Prepare prompt and run inference
    prompt = f"Question: {question} Answer:"
    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    ).to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=3,
        )

    prediction = processor.decode(outputs[0], skip_special_tokens=True).strip()

    # Exact match check
    is_correct = prediction.strip().lower() == ground_truth.strip().lower()
    if is_correct:
        correct += 1
    total += 1

    result = {
        "img_id": row["img_id"],
        "complexity": int(row["complexity"]),
        "question_class": row["question_class"],
        "question": question,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "exact_match": is_correct,
    }
    results.append(result)

    # Print result
    status = "CORRECT ✓" if is_correct else "WRONG ✗"
    print(f"[{idx+1}/{len(sample_df)}] {status}")
    print(f"  Question:   {question[:100]}")
    print(f"  GT Answer:  {ground_truth[:80]}")
    print(f"  Predicted:  {prediction[:80]}")
    print("-" * 80)

## 7. Results Summary

In [ ]:
import json

accuracy = (correct / total * 100) if total > 0 else 0
results_df = pd.DataFrame(results)

print(f"{'=' * 60}")
print(f"BASELINE INFERENCE SUMMARY")
print(f"{'=' * 60}")
print(f"  Model:           {MODEL_NAME}")
print(f"  Total samples:   {total}")
print(f"  Exact matches:   {correct}")
print(f"  Accuracy:        {accuracy:.1f}%")
print(f"\n  Per-Complexity Accuracy:")
for c in sorted(results_df["complexity"].unique()):
    c_df = results_df[results_df["complexity"] == c]
    c_acc = c_df["exact_match"].mean() * 100
    print(f"    Level {c}: {c_df['exact_match'].sum()}/{len(c_df)} ({c_acc:.1f}%)")
print(f"{'=' * 60}")

# Save predictions CSV
results_path = os.path.join(RESULTS_DIR, "baseline_predictions.csv")
results_df.to_csv(results_path, index=False)
print(f"\nPredictions saved to {results_path}")

# Save summary JSON
summary = {
    "model": MODEL_NAME,
    "num_samples": total,
    "exact_match_accuracy": accuracy,
    "correct": int(correct),
    "total": int(total),
    "per_complexity": {}
}
for c in sorted(results_df["complexity"].unique()):
    c_df = results_df[results_df["complexity"] == c]
    summary["per_complexity"][f"level_{c}"] = {
        "correct": int(c_df["exact_match"].sum()),
        "total": int(len(c_df)),
        "accuracy": round(c_df["exact_match"].mean() * 100, 1)
    }

summary_path = os.path.join(RESULTS_DIR, "baseline_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to {summary_path}")

## 8. Visualize Sample Predictions

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Show first 6 predictions with images
n_show = min(6, len(results))
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i in range(n_show):
    r = results[i]
    img_path = os.path.join(IMAGE_DIR, f"{r['img_id']}.jpg")
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_axis_off()

    status = "✓ CORRECT" if r["exact_match"] else "✗ WRONG"
    color = "green" if r["exact_match"] else "red"

    q_short = r["question"][:70] + ("..." if len(r["question"]) > 70 else "")
    gt_short = r["ground_truth"][:50] + ("..." if len(r["ground_truth"]) > 50 else "")
    pred_short = r["prediction"][:50] + ("..." if len(r["prediction"]) > 50 else "")

    axes[i].set_title(
        f"{status} | Complexity {r['complexity']}\n"
        f"Q: {q_short}\n"
        f"GT: {gt_short}\n"
        f"Pred: {pred_short}",
        fontsize=8, loc="left", color=color if status.startswith("✗") else "black"
    )

for i in range(n_show, len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Baseline BLIP-2 Zero-Shot Predictions on Kvasir-VQA-x1", fontsize=14)
plt.tight_layout()

viz_path = os.path.join(RESULTS_DIR, "baseline_predictions_viz.png")
plt.savefig(viz_path, dpi=150)
plt.show()
print(f"Visualization saved to {viz_path}")